# Notebook 3/4 — LightAlzNet Training (novel model)
**Time:** ~35 min on free T4  
**Output:** `checkpoints/lightalznet_best.pth`

## Before running
1. Runtime → Change runtime type → **T4 GPU**
2. Run Cell 1 (install) → restart runtime
3. Run all remaining cells top to bottom
4. When done, you can close this tab — the checkpoint is saved to Drive

> After this finishes, open **Notebook 4** for evaluation + all thesis outputs.


In [1]:
import warnings; warnings.filterwarnings('ignore')
!pip install "numpy<2.0.0" "scipy<1.13.0" "scikit-learn<1.5.0" \
             "pandas==2.2.2" nibabel matplotlib pillow torchinfo --quiet
print("✓ Done. Restart runtime now.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 108.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but 

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✓ Drive mounted')


Mounted at /content/drive
✓ Drive mounted


In [2]:
import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
import numpy as np, pandas as pd, json, time, os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, balanced_accuracy_score
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import nibabel as nib
from PIL import Image
import torchvision.transforms as T

# ── Paths ────────────────────────────────────────────────────────────────
BASE   = Path('/content/drive/MyDrive/alzheimer_thesis')
O3_DIR = BASE / 'data' / 'oasis3_raw'
CKPT   = BASE / 'checkpoints'
LOGS   = BASE / 'logs'
MANI   = BASE / 'data' / 'manifests'
GRAD   = BASE / 'gradcam'
COMP   = BASE / 'compression'
for d in [CKPT, LOGS, MANI, GRAD, COMP]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224; NUM_CLASSES = 3
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')


Device: cuda
GPU: Tesla T4


In [3]:
def extract_25d(nifti_path, size=IMG_SIZE):
    vol = nib.load(str(nifti_path)).get_fdata().astype(np.float32)
    if vol.ndim == 4: vol = vol[...,0]
    z, mid = vol.shape[2], vol.shape[2]//2
    slices = []
    for off in [-1,0,1]:
        idx = int(np.clip(mid+off, 0, z-1))
        sl  = vol[:,:,idx]
        sl  = np.array(Image.fromarray(sl).resize((size,size), Image.BILINEAR))
        sl  = (sl - sl.mean()) / (sl.std() + 1e-8)
        slices.append(sl.astype(np.float32))
    return np.stack(slices, axis=0)


In [4]:
class OASIS3Dataset(Dataset):
    def __init__(self, csv_path, augment=False):
        self.df = pd.read_csv(csv_path)
        self.aug = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=15),
            T.RandomAffine(degrees=0, translate=(0.08,0.08), scale=(0.9,1.1)),
            T.GaussianBlur(kernel_size=3, sigma=(0.1,1.0)),
        ]) if augment else None
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        t = torch.from_numpy(extract_25d(row['path']))
        if self.aug: t = self.aug(t)
        return t, int(row['label'])

def make_sampler(df):
    counts = df['label'].value_counts().sort_index().values
    w = 1.0/counts
    return WeightedRandomSampler([w[int(l)] for l in df['label']], len(df), replacement=True)

BATCH = 16
train_df = pd.read_csv(MANI/'train.csv')
train_loader = DataLoader(OASIS3Dataset(MANI/'train.csv', augment=True), batch_size=BATCH,
                          sampler=make_sampler(train_df), num_workers=2, pin_memory=True)
val_loader   = DataLoader(OASIS3Dataset(MANI/'val.csv'),  batch_size=BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(OASIS3Dataset(MANI/'test.csv'), batch_size=BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)
x, y = next(iter(train_loader))
print(f'Data OK: batch {tuple(x.shape)}, range [{x.min():.2f},{x.max():.2f}]')


Data OK: batch (16, 3, 224, 224), range [-1.07,4.47]


In [5]:
# ── EfficientNet-B0 ──────────────────────────────────────────────────────
def build_efficientnet():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(m.classifier[1].in_features,128),
                                  nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, NUM_CLASSES))
    for p in m.features.parameters(): p.requires_grad = False
    return m

# ── MobileNetV3-Small ────────────────────────────────────────────────────
def build_mobilenet():
    m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    in_f = m.classifier[3].in_features
    m.classifier[3] = nn.Sequential(nn.Linear(in_f,64), nn.ReLU(), nn.Dropout(0.4),
                                     nn.Linear(64, NUM_CLASSES))
    for p in m.features.parameters(): p.requires_grad = False
    return m

# ── LightAlzNet ──────────────────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self,ch,r=16):
        super().__init__()
        self.se=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Linear(ch,max(ch//r,4),bias=False),nn.ReLU(),
            nn.Linear(max(ch//r,4),ch,bias=False),nn.Sigmoid())
    def forward(self,x): return x*self.se(x).view(x.size(0),-1,1,1)

class DWSConvBlock(nn.Module):
    def __init__(self,in_ch,out_ch,stride=1):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(in_ch,in_ch,3,stride=stride,padding=1,groups=in_ch,bias=False),
            nn.BatchNorm2d(in_ch),nn.ReLU6(inplace=True),
            nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU6(inplace=True))
        self.se=SEBlock(out_ch)
        self.skip=(nn.Conv2d(in_ch,out_ch,1,stride=stride,bias=False)
                   if (in_ch!=out_ch or stride!=1) else nn.Identity())
    def forward(self,x): return self.se(self.block(x))+self.skip(x)

class LightAlzNet(nn.Module):
    def __init__(self,num_classes=3):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv2d(3,32,3,stride=2,padding=1,bias=False),
                                  nn.BatchNorm2d(32),nn.ReLU6(inplace=True))
        self.body=nn.Sequential(DWSConvBlock(32,64,2),DWSConvBlock(64,128,2),
            DWSConvBlock(128,128,1),DWSConvBlock(128,256,2),
            DWSConvBlock(256,256,1),DWSConvBlock(256,512,2))
        self.head=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),
                                  nn.Linear(512,256),nn.ReLU(),nn.Dropout(0.5),
                                  nn.Linear(256,num_classes))
    def forward(self,x): return self.head(self.body(self.stem(x)))


In [6]:
class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,alpha=None,ls=0.1):
        super().__init__(); self.gamma,self.alpha,self.ls=gamma,alpha,ls
    def forward(self,logits,targets):
        n=logits.size(1)
        smooth=torch.full_like(logits,self.ls/(n-1))
        smooth.scatter_(1,targets.unsqueeze(1),1.0-self.ls)
        log_p=F.log_softmax(logits,dim=1); p=torch.exp(log_p)
        loss=-((1-p)**self.gamma*smooth*log_p).sum(dim=1)
        if self.alpha is not None: loss=loss*self.alpha[targets]
        return loss.mean()

def get_weights(device):
    df=pd.read_csv(MANI/'train.csv')
    counts=df['label'].value_counts().sort_index().values.astype(float)
    w=1.0/counts; w=w/w.sum()*len(counts)
    return torch.tensor(w,dtype=torch.float32).to(device)

def evaluate(model,loader,device):
    model.eval(); preds,labels,probs=[],[],[]
    with torch.no_grad():
        for x,y in loader:
            p=torch.softmax(model(x.to(device)),dim=1)
            probs.extend(p.cpu().numpy()); preds.extend(p.argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    preds,labels,probs=np.array(preds),np.array(labels),np.array(probs)
    return {'acc':(preds==labels).mean(),
            'macro_f1':f1_score(labels,preds,average='macro',zero_division=0),
            'bal_acc':balanced_accuracy_score(labels,preds),
            'preds':preds,'labels':labels,'probs':probs}

def unfreeze_last_blocks(model, model_name, n=2):
    if model_name in ('efficientnet','mobilenet'):
        blocks=list(model.features.children())
        for b in blocks[-n:]:
            for p in b.parameters(): p.requires_grad=True
        unfrozen=sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
        print(f'  Phase 2: unfroze last {n} blocks → trainable={unfrozen:.2f}M')

def train_one_model(model, model_name, lr=3e-4, epochs=80, patience=20, is_pretrained=True):
    """
    Train a single model with two-phase strategy.
    Saves checkpoint on every improvement. Fully resumable.
    Expected time on free T4: ~30-40 min.
    """
    criterion = FocalLoss(gamma=2.0, alpha=get_weights(DEVICE), ls=0.1)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=lr, weight_decay=5e-2)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/100)

    ckpt_path = CKPT / f'{model_name}_best.pth'
    best_f1, no_improve, history = 0.0, 0, []
    start_epoch = 0
    phase2_done = False

    # Resume if checkpoint exists
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck['model'])
        best_f1     = ck.get('best_f1', 0)
        start_epoch = ck.get('epoch', 0)
        # If we already passed epoch 15, phase 2 was done — unfreeze now
        if is_pretrained and start_epoch >= 15:
            unfreeze_last_blocks(model, model_name)
            phase2_done = True
            optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=lr/10, weight_decay=5e-2)
            scheduler = CosineAnnealingLR(optimizer, T_max=max(epochs-start_epoch,1), eta_min=lr/1000)
        print(f'  Resumed from epoch {start_epoch} | best F1={best_f1:.4f}')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
    total_p   = sum(p.numel() for p in model.parameters())/1e6
    print(f'Training {model_name} | {trainable:.2f}M trainable / {total_p:.2f}M total | lr={lr}')
    model.to(DEVICE); t0=time.time()

    for epoch in range(start_epoch, epochs):
        # Phase 2 trigger
        if is_pretrained and epoch == 15 and not phase2_done:
            unfreeze_last_blocks(model, model_name)
            phase2_done = True
            optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=lr/10, weight_decay=5e-2)
            scheduler = CosineAnnealingLR(optimizer, T_max=epochs-15, eta_min=lr/1000)

        # Warmup
        warmup_ep = epoch if not phase2_done else epoch-15
        if warmup_ep < 5:
            cur = (lr if not phase2_done else lr/10)*(warmup_ep+1)/5
            for pg in optimizer.param_groups: pg['lr']=cur

        model.train(); tl,cor,tot=0.0,0,0
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE); optimizer.zero_grad()
            out=model(x); loss=criterion(out,y); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step()
            tl+=loss.item()*x.size(0); cor+=(out.argmax(1)==y).sum().item(); tot+=x.size(0)

        if warmup_ep>=5: scheduler.step()
        val=evaluate(model, val_loader, DEVICE)
        history.append({'epoch':epoch+1,'train_loss':round(tl/tot,4),
                        'train_acc':round(cor/tot,4),'val_acc':round(val['acc'],4),
                        'val_f1':round(val['macro_f1'],4)})

        tag='P2' if phase2_done else 'P1'
        print(f'  [{tag}] Ep {epoch+1:>3}/{epochs} | '
              f'loss={tl/tot:.4f} | tr={cor/tot:.3f} | '
              f'val={val["acc"]:.3f} | F1={val["macro_f1"]:.3f} | '
              f'{(time.time()-t0)/60:.1f}min')

        if val['macro_f1'] > best_f1:
            best_f1, no_improve = val['macro_f1'], 0
            torch.save({'model':model.state_dict(),'best_f1':best_f1,'epoch':epoch+1}, ckpt_path)
            print(f'  ★ Best F1={best_f1:.4f} — saved')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch+1}')
                break

    (LOGS/f'{model_name}_history.json').write_text(json.dumps(history))
    print(f'\n✓ Done | {model_name} | Best val F1={best_f1:.4f}')
    print(f'  Checkpoint: {ckpt_path}')
    return best_f1


In [7]:
# ── Train LightAlzNet (from scratch) ─────────────────────────────────────
# LightAlzNet has no pretrained weights, so:
# - is_pretrained=False (no frozen backbone, no phase 2)
# - lr=1e-3 (higher — needs to learn from scratch faster)
print('='*55)
print('LightAlzNet | novel architecture | trains from scratch')
print('='*55)
model = LightAlzNet(num_classes=3).to(DEVICE)
best_f1 = train_one_model(model, 'lightalznet', lr=1e-3, epochs=80, patience=20, is_pretrained=False)
print(f'\n✓ LightAlzNet complete | best val F1 = {best_f1:.4f}')
print('  → Open Notebook 4 for evaluation + thesis outputs')


LightAlzNet | novel architecture | trains from scratch
  Resumed from epoch 49 | best F1=0.5056
Training lightalznet | 0.63M trainable / 0.63M total | lr=0.001
  [P1] Ep  50/80 | loss=0.4399 | tr=0.522 | val=0.386 | F1=0.382 | 0.9min
  [P1] Ep  51/80 | loss=0.4405 | tr=0.473 | val=0.386 | F1=0.344 | 1.7min
  [P1] Ep  52/80 | loss=0.4286 | tr=0.527 | val=0.432 | F1=0.435 | 2.6min
  [P1] Ep  53/80 | loss=0.4319 | tr=0.512 | val=0.364 | F1=0.292 | 3.4min
  [P1] Ep  54/80 | loss=0.4037 | tr=0.580 | val=0.432 | F1=0.403 | 4.2min
  [P1] Ep  55/80 | loss=0.4202 | tr=0.527 | val=0.341 | F1=0.337 | 5.1min
  [P1] Ep  56/80 | loss=0.4337 | tr=0.517 | val=0.409 | F1=0.334 | 5.9min
  [P1] Ep  57/80 | loss=0.4207 | tr=0.546 | val=0.364 | F1=0.286 | 6.7min
  [P1] Ep  58/80 | loss=0.4402 | tr=0.429 | val=0.295 | F1=0.207 | 7.6min
  [P1] Ep  59/80 | loss=0.4062 | tr=0.546 | val=0.432 | F1=0.394 | 8.4min
  [P1] Ep  60/80 | loss=0.4308 | tr=0.473 | val=0.341 | F1=0.172 | 9.2min
  [P1] Ep  61/80 | loss=0.